# Estudiando las frases

In [10]:
import pandas as pd

In [11]:
df = pd.read_pickle('/home/user/work/Preprocessing/MSKA/Dataset/data/Phoenix-2014T.train.pkl')
rows = []

for k, v in df.items():
    rows.append({
        "id": k,
        "name": v["name"],
        "gloss": v["gloss"],
        "text": v["text"],
        "num_frames": v["num_frames"],
        "keypoint_shape": tuple(v["keypoint"].shape)
    })

df = pd.DataFrame(rows)
df.head()

,id,name,gloss,text,num_frames,keypoint_shape
0,11August_2010_Wednesday_tagesschau-1,11August_2010_Wednesday_tagesschau-1,JETZT WETTER MORGEN DONNERSTAG ZWOELF FEBRUAR,und nun die wettervorhersage für morgen donner...,86,"(86, 136, 3)"
1,25October_2010_Monday_tagesschau-15,25October_2010_Monday_tagesschau-15,ITALIEN IX TIEF DRUCK KOMMEN HEUTE NACHT BERG ...,das tief über italien sorgt dafür dass es an d...,138,"(138, 136, 3)"
2,11August_2010_Wednesday_tagesschau-7,11August_2010_Wednesday_tagesschau-7,WOLKE LOCH SPEZIELL NORDWEST,größere wolkenlücken finden sich vor allem im ...,71,"(71, 136, 3)"
3,11August_2010_Wednesday_tagesschau-9,11August_2010_Wednesday_tagesschau-9,FLUSS HEUTE NACHT SECHS FLUSS SIEBZEHN GRAD,im emsland heute nacht nur neun am oberrhein b...,105,"(105, 136, 3)"
4,11August_2010_Wednesday_tagesschau-13,11August_2010_Wednesday_tagesschau-13,TEMPERATUR BLEIBEN GLEICH,am temperaturniveau ändert sich wenig,48,"(48, 136, 3)"


In [15]:
df['num_frames'].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99])

count    7096.000000
mean      116.594419
std        49.859426
min        16.000000
50%       112.000000
75%       144.000000
90%       182.000000
95%       208.000000
99%       259.000000
max       475.000000
Name: num_frames, dtype: float64

In [17]:
from transformers import AutoTokenizer
import warnings
warnings.filterwarnings("ignore")

# Cargar el tokenizador
tokenizer_path = "Qwen/Qwen2.5-1.5B"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, trust_remote_code=True)

# Función para contar tokens
def count_qwen_tokens(text):
    if not isinstance(text, str):
        return 0
    # Tokenizamos sin truncar para ver la longitud real completa
    return len(tokenizer(text, truncation=False)['input_ids'])

print("Calculando longitudes de tokens...")

# Aplicar la función a las columnas de Texto y Glosas
df['text_tokens'] = df['text'].apply(count_qwen_tokens)


# Mostrar las estadísticas con percentiles
print("\n" + "="*50)
print("📊 ESTADÍSTICAS DE TOKENS DE TEXTO (Alemán)")
print("="*50)
print(df['text_tokens'].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]))

# Ver la relación Fotogramas vs Tokens
df['ratio_frames_text'] = df['num_frames'] / df['text_tokens']
print("\n" + "="*50)
print("RATIO: Fotogramas de video por cada Token de texto")
print("="*50)
print(df['ratio_frames_text'].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]))

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Calculando longitudes de tokens...

📊 ESTADÍSTICAS DE TOKENS DE TEXTO (Alemán)
count    7096.000000
mean       23.590614
std         9.722534
min         3.000000
50%        22.000000
75%        29.000000
90%        37.000000
95%        41.000000
99%        51.000000
max        78.000000
Name: text_tokens, dtype: float64

RATIO: Fotogramas de video por cada Token de texto
count    7096.000000
mean        5.028114
std         1.179292
min         0.551724
50%         4.951800
75%         5.619048
90%         6.402703
95%         6.963602
99%         8.447222
max        18.800000
Name: ratio_frames_text, dtype: float64


In [15]:
df

,id,name,gloss,text,num_frames,keypoint_shape,text_tokens,ratio_frames_text
0,11August_2010_Wednesday_tagesschau-1,11August_2010_Wednesday_tagesschau-1,JETZT WETTER MORGEN DONNERSTAG ZWOELF FEBRUAR,und nun die wettervorhersage für morgen donner...,86,"(86, 136, 3)",19,4.526316
1,25October_2010_Monday_tagesschau-15,25October_2010_Monday_tagesschau-15,ITALIEN IX TIEF DRUCK KOMMEN HEUTE NACHT BERG ...,das tief über italien sorgt dafür dass es an d...,138,"(138, 136, 3)",26,5.307692
2,11August_2010_Wednesday_tagesschau-7,11August_2010_Wednesday_tagesschau-7,WOLKE LOCH SPEZIELL NORDWEST,größere wolkenlücken finden sich vor allem im ...,71,"(71, 136, 3)",16,4.437500
3,11August_2010_Wednesday_tagesschau-9,11August_2010_Wednesday_tagesschau-9,FLUSS HEUTE NACHT SECHS FLUSS SIEBZEHN GRAD,im emsland heute nacht nur neun am oberrhein b...,105,"(105, 136, 3)",21,5.000000
4,11August_2010_Wednesday_tagesschau-13,11August_2010_Wednesday_tagesschau-13,TEMPERATUR BLEIBEN GLEICH,am temperaturniveau ändert sich wenig,48,"(48, 136, 3)",10,4.800000
...,...,...,...,...,...,...,...,...
7091,27January_2013_Sunday_tagesschau-8837,27January_2013_Sunday_tagesschau-8837,MORGEN TATSAECHLICH FROST negalp-KEIN EINS ACHT,morgen seit längerem wieder frostfrei ein grad...,104,"(104, 136, 3)",23,4.521739
7092,27January_2013_Sunday_tagesschau-8831,27January_2013_Sunday_tagesschau-8831,HEUTE NACHT REGION SCHNEE REGEN VORSICHT GLATT...,heute nacht gibt es im osten und süden noch sc...,115,"(115, 136, 3)",32,3.593750
7093,27January_2013_Sunday_tagesschau-8841,27January_2013_Sunday_tagesschau-8841,DONNERSTAG FREUNDLICH SONNE DANN SPAETER KOMME...,der donnerstag beginnt oft freundlich später z...,77,"(77, 136, 3)",18,4.277778
7094,22July_2010_Thursday_heute-8812,22July_2010_Thursday_heute-8812,MAXIMAL TEMPERATUR UNGEFAEHR FUENF ZWANZIG GRA...,die höchsttemperatur die wird so etwa bei fünf...,179,"(179, 136, 3)",32,5.593750


In [21]:
# Función adaptada a la lógica exacta de tu GlossTokenizer_S2G
def count_gloss_tokens(gloss_text):
    if not isinstance(gloss_text, str):
        return 0
    # Como vimos en tu código (__call__), tú separas por espacios: gls_seq.split()
    # Además, filtramos posibles espacios extra o nulos
    tokens = [gls for gls in gloss_text.split() if gls.strip()]
    return len(tokens)

print("Calculando longitudes de tokens de Glosas...")

# Aplicar la función a la columna de glosas
df['gloss_tokens'] = df['gloss'].apply(count_gloss_tokens)

# Estadísticas de Glosas
print("\n" + "="*50)
print("📊 ESTADÍSTICAS DE TOKENS DE GLOSAS (Phoenix)")
print("="*50)
print(df['gloss_tokens'].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]))

# Relación Glosas vs Texto (Compresión de la traducción)
# Cuidado con las divisiones por cero si hay glosas vacías
df['ratio_gloss_text'] = df['gloss_tokens'] / df['text_tokens'].replace(0, 1)
print("\n" + "="*50)
print("RATIO: Tokens de Glosa por cada Token de texto de Qwen")
print("="*50)
print(df['ratio_gloss_text'].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]))

# Relación Fotogramas vs Glosas (Velocidad de la persona signante)
df['ratio_frames_gloss'] = df['num_frames'] / df['gloss_tokens'].replace(0, 1)
print("\n" + "="*50)
print("RATIO: Fotogramas de vídeo por cada Glosa")
print("="*50)
print(df['ratio_frames_gloss'].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]))

Calculando longitudes de tokens de Glosas...

📊 ESTADÍSTICAS DE TOKENS DE GLOSAS (Phoenix)
count    7096.000000
mean        7.785654
std         3.436425
min         1.000000
50%         7.000000
75%        10.000000
90%        12.000000
95%        14.000000
99%        18.000000
max        30.000000
Name: gloss_tokens, dtype: float64

RATIO: Tokens de Glosa por cada Token de texto de Qwen
count    7096.000000
mean        0.339451
std         0.097696
min         0.058824
50%         0.333333
75%         0.388889
90%         0.461538
95%         0.500000
99%         0.625000
max         1.500000
Name: ratio_gloss_text, dtype: float64

RATIO: Fotogramas de vídeo por cada Glosa
count    7096.000000
mean       15.516105
std         4.277521
min         3.200000
50%        14.839744
75%        17.430804
90%        20.428571
95%        22.800000
99%        29.333333
max        71.500000
Name: ratio_frames_gloss, dtype: float64
